In [1]:
from datetime import datetime, timezone

import cmasher as cmr
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.font_manager import FontProperties
from mpl_toolkits.basemap import Basemap

from skyweaver import Observatory, OrbitSpec, TimeGrid, ground_track

In [21]:
orbit = OrbitSpec.circular(
    name="test_550km_i70",
    epoch=datetime(2026, 1, 1, tzinfo=timezone.utc),
    altitude_km=550.0,
    inclination_deg=70.0,
    raan_deg=30.0,
    phase_deg=120.0,
)

site = Observatory.get("MWA")

grid = TimeGrid(
    start=datetime(2026, 1, 1, 0, 39, 0, tzinfo=timezone.utc),
    stop=datetime(2026, 1, 2, 0, 39, 0, tzinfo=timezone.utc),
    cadence_s=60.0,
)

gt = ground_track(orbit, grid)

In [ ]:
bitter_title = FontProperties(fname="/Users/achokshi/Desktop/Bitter/static/Bitter-Italic.ttf")

lon = gt.longitude_deg.copy()
lat = gt.latitude_deg.copy()

# find jumps across the dateline
jumps = np.abs(np.diff(lon)) > 180.0

# insert NaNs after jump locations
lon_plot = lon.astype(float)
lat_plot = lat.astype(float)

lon_plot = np.insert(lon_plot, np.where(jumps)[0] + 1, np.nan)
lat_plot = np.insert(lat_plot, np.where(jumps)[0] + 1, np.nan)

for i in range(gt.longitude_deg.size):
    fig, ax = plt.subplots(figsize=(10, 5))

    m = Basemap(
        projection="cyl",
        llcrnrlat=-86,
        urcrnrlat=86,
        llcrnrlon=-180,
        urcrnrlon=180,
        resolution="c",
        ax=ax,
    )
    m.drawcoastlines(color="#222222", linewidth=0.3)
    m.drawmapboundary(fill_color=cmr.pride(0.3))
    m.fillcontinents(color=cmr.pride(0.73), lake_color=cmr.pride(0.3))
    m.nightshade(grid.datetimes()[i], color=cmr.pride(0.1), delta=0.1)

    observatories = [
        Observatory.get("MWA"),
        Observatory.get("LOFAR"),
        Observatory.get("HERA"),
        Observatory.get("ALBATROS"),
        Observatory.get("CHORD"),
    ]

    for obs in observatories:
        x, y = m(obs.longitude_deg, obs.latitude_deg)
        x = float(str(x))
        y = float(str(y))

        ax.scatter(
            x,
            y,
            s=121,
            marker=r"$⬢$",
            color=cmr.pride(0.6),
            edgecolors="whitesmoke",
            linewidths=1,
            zorder=49,
        )

    ax.plot(
        lon_plot[:i],
        lat_plot[:i],
        lw=1,
        alpha=1,
        color="whitesmoke",
    )

    ax.scatter(
        lon_plot[i],
        lat_plot[i],
        marker=r"$✠$",
        s=384,
        color="white",
        ec="#222222",
        lw=0.2,
        zorder=99,
    )

    title = "  skyweaver  "

    x0, y0 = 0.016, 0.02

    ax.text(
        x0 + 0.015,
        y0 + 0.045,
        title,
        transform=ax.transAxes,
        fontsize=36,
        fontproperties=bitter_title,
        color="whitesmoke",
        ha="left",
        va="bottom",
        zorder=201,
        bbox=dict(
            boxstyle="square,pad=0.2",
            facecolor="#222222",
            edgecolor="whitesmoke",
            linewidth=1.0,
            alpha=0.9,
        ),
    )

    plt.savefig(f"logo/{i}.png", dpi=300)
    plt.close()